In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_curve, auc, confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

In [ ]:
# Ângulos de mistura
sin2_theta12 = 0.307
sin2_theta23 = 0.56
sin2_theta13 = 0.022

s12 = np.sqrt(sin2_theta12)
c12 = np.sqrt(1 - sin2_theta12)
s23 = np.sqrt(sin2_theta23)
c23 = np.sqrt(1 - sin2_theta23)
s13 = np.sqrt(sin2_theta13)
c13 = np.sqrt(1 - sin2_theta13)

# Diferenças de massa
Delta_m21 = 7.53e-5
Delta_m31 = 2.51e-3

r_delta = Delta_m21/Delta_m31

# Fator de Jarlskog
Jr = c12*s12*c23*s23*s13

p_mu = 27

rho = 2.5  # g/cm^3
Ye = 0.5

def matter_potential(E):
    return 7.56e-5*rho*Ye*E

In [ ]:
def probability_sm(E, L, delta_cp):

    Delta = 1.267 * Delta_m31 * L / E

    a = matter_potential(E)

    ra = a / Delta_m31

    term1 = (
        4*s13**2 * s23**2
        /(1 - ra)**2
        *np.sin((1 - ra)*Delta)**2
    )

    term2 = (
        8*Jr*r_delta
        /(ra*(1 - ra))
        *np.cos(delta_cp + Delta)
        *np.sin(ra*Delta)
        *np.sin((1 - ra)*Delta)
    )

    P = term1 + term2

    return np.clip(P, 0, 1)

In [ ]:
def probability_nsi(E, L, delta_cp, epsilon, phi_nsi):

    Delta = 1.267*Delta_m31*L/E
    a = matter_potential(E)
    ra = a/Delta_m31

    # Termo padrão
    term_sm = probability_sm(E, L, delta_cp)

    # Termo quadrático em epsilon
    term_eps2 = p_mu**2 * epsilon**2

    # Termo de interferência
    term_interf = (
        4*p_mu*epsilon
        *s13*s23
        /(1 - ra)
        *np.sin((1 - ra)*Delta)
        *np.sin(delta_cp - phi_nsi + (1 - ra)*Delta)
    )

    P = term_sm + term_eps2 + term_interf

    return np.clip(P, 0, 1)

In [ ]:
E_vals = np.linspace(0.2, 10, 1000)
L_fixed = 1000 #km
delta_cp = 1.2*np.pi
phi_nsi = 0.5*np.pi
epsilon = 2e-3

P_sm = probability_sm(E_vals, L_fixed, delta_cp)
P_nsi = probability_nsi(E_vals, L_fixed, delta_cp, epsilon, phi_nsi)

plt.figure(figsize=(10,6))
plt.plot(E_vals, P_sm, label='SM')
plt.plot(E_vals, P_nsi, label='NSI')
plt.xlabel('Energy [GeV]')
plt.ylabel('Probability')
plt.title('Oscillation Probability')
plt.legend()
plt.grid()
plt.savefig('plot1.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

for eps in [0, 1e-4, 1e-3, 2e-3, 1e-2]:

    P = probability_nsi(E_vals, L_fixed, delta_cp, eps, phi_nsi)

    plt.plot(E_vals, P, label=f'epsilon={eps:.1e}')

plt.xlabel('Energy [GeV]')
plt.ylabel('Probability')
plt.title('Dependence on Epsilon')
plt.legend()
plt.grid()
plt.savefig('plot2.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

for phi in [0, np.pi/2, np.pi, 3*np.pi/2]:

    P = probability_nsi(E_vals, L_fixed, delta_cp, epsilon, phi)

    plt.plot(E_vals, P, label=f'phi={phi/np.pi:.1f}π')

plt.xlabel('Energy [GeV]')
plt.ylabel('Probability')
plt.title('Dependence in the NSI phase')
plt.legend()
plt.grid()
plt.savefig('plot3.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
N = 50000

energia = np.random.uniform(0.2, 10, N)

# distribuição log-uniforme em baseline
logL = np.random.uniform(np.log10(20), np.log10(12700), N)
L = 10**logL

# fases aleatórias
delta_cp_vals = np.random.uniform(0, 2*np.pi, N)
phi_vals = np.random.uniform(0, 2*np.pi, N)

# classe SM
P_sm_events = probability_sm(energia, L, delta_cp_vals)

# classe NSI
P_nsi_events = probability_nsi(energia, L, delta_cp_vals, epsilon=1e-3, phi_nsi=phi_vals)

In [ ]:
sigma_E = 0.03
sigma_P = 0.001

energia_sm = energia + np.random.normal(0, sigma_E, N)
energia_nsi = energia + np.random.normal(0, sigma_E, N)

P_sm_events += np.random.normal(0, sigma_P, N)
P_nsi_events += np.random.normal(0, sigma_P, N)

P_sm_events = np.clip(P_sm_events, 0, 1)
P_nsi_events = np.clip(P_nsi_events, 0, 1)

In [ ]:
sm_df = pd.DataFrame({
    'energia': energia_sm,
    'baseline': L,
    'L_over_E': L/energia_sm,
    'probabilidade': P_sm_events,
    'classe': 0
})

nsi_df = pd.DataFrame({
    'energia': energia_nsi,
    'baseline': L,
    'L_over_E': L/energia_nsi,
    'probabilidade': P_nsi_events,
    'classe': 1
})

df = pd.concat([sm_df, nsi_df])
print(df.head())

In [ ]:
plt.figure(figsize=(10,6))

plt.hist(
    sm_df['probabilidade'],
    bins=60,
    alpha=0.5,
    density=True,
    label='SM'
)

plt.hist(
    nsi_df['probabilidade'],
    bins=60,
    alpha=0.5,
    density=True,
    label='NSI'
)
plt.xlim(-0.005,0.2)
plt.xlabel('Probability')
plt.ylabel('Density')
plt.title('Probability Distribution')
plt.legend()
plt.grid()
plt.savefig('plot4.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(
    sm_df['energia'][:4000],
    sm_df['probabilidade'][:4000],
    s=5,
    alpha=0.4,
    label='SM'
)

plt.scatter(
    nsi_df['energia'][:4000],
    nsi_df['probabilidade'][:4000],
    s=5,
    alpha=0.4,
    label='NSI'
)

plt.xlabel('Energy [GeV]')
plt.ylabel('Probability')
plt.title('Energy vs Probability')
plt.legend()
plt.grid()
plt.savefig('plot5.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(
    sm_df['L_over_E'][:4000],
    sm_df['probabilidade'][:4000],
    s=5,
    alpha=0.4,
    label='SM'
)

plt.scatter(
    nsi_df['L_over_E'][:4000],
    nsi_df['probabilidade'][:4000],
    s=5,
    alpha=0.4,
    label='NSI'
)

plt.xscale('log')
plt.xlabel('L/E [km/GeV]')
plt.ylabel('Probability')
plt.title('Oscillatory Structure in L/E')
plt.legend()
plt.grid()
plt.savefig('plot6.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
features = [
    'energia',
    'baseline',
    'L_over_E',
    'probabilidade'
]

X = df[features]
y = df['classe']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    random_state=42
)

model.fit(X_train_scaled, y_train)

In [ ]:
probs = model.predict_proba(X_test_scaled)[:,1]
pred = model.predict(X_test_scaled)

In [ ]:
fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8,8))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid()
plt.savefig('plot7.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(6,6))
plt.imshow(cm)
plt.xticks([0,1], ['SM','NSI'])
plt.yticks([0,1], ['SM','NSI'])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha='center', va='center')

plt.title('Confusion Matrix')
plt.colorbar()
plt.savefig('plot8.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
print(classification_report(y_test, pred))

In [ ]:
importance = model.feature_importances_

plt.figure(figsize=(8,6))
plt.bar(features, importance)
plt.ylabel('Importance')
plt.title('Importance of features')
plt.grid()
plt.savefig('plot9.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
E_grid = np.linspace(0.1, 10, 200)
L_grid = np.linspace(20, 12700, 200)

EE, LL = np.meshgrid(E_grid, L_grid)

P_grid = probability_nsi(EE, LL, delta_cp=1.2*np.pi, epsilon=1e-3, phi_nsi=1.5*np.pi)

plt.figure(figsize=(10,7))
plt.contourf(EE, LL, P_grid, levels=100)

plt.xlabel('Energy [GeV]')
plt.ylabel('Baseline [km]')
plt.title('NSI Probability Map')
plt.colorbar(label='P(νμ → νe)')
plt.savefig('plot10.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
P_grid_sm = probability_sm(EE, LL, delta_cp=1.2*np.pi)

DeltaP = P_grid - P_grid_sm

plt.figure(figsize=(10,7))

plt.contourf(EE, LL, DeltaP, levels=100)

plt.xlabel('Energy [GeV]')
plt.ylabel('Baseline [km]')
plt.title('Diference Between NSI and SM')
plt.colorbar(label='ΔP')
plt.savefig('plot11.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
epsilons = np.linspace(0, 2e-3, 12)

auc_list = []

for eps in epsilons:

    P_nsi_tmp = probability_nsi(energia, L, delta_cp_vals, epsilon=eps, phi_nsi=phi_vals)

    nsi_tmp = pd.DataFrame({
        'energia': energia,
        'baseline': L,
        'L_over_E': L/energia,
        'probabilidade': P_nsi_tmp,
        'classe': 1
    })

    sm_tmp = pd.DataFrame({
        'energia': energia,
        'baseline': L,
        'L_over_E': L/energia,
        'probabilidade': P_sm_events,
        'classe': 0
    })

    df_tmp = pd.concat([sm_tmp, nsi_tmp])

    X = df_tmp[features]
    y = df_tmp['classe']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )

    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_val = auc(fpr, tpr)
    auc_list.append(auc_val)

print(auc_list)

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(epsilons, auc_list, 'o-')
plt.xlabel('Epsilon')
plt.ylabel('ROC AUC')
plt.title('ROC Sensitivity to the NSI Parameter')
plt.grid()
plt.savefig('plot12.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
event_sizes = [1000, 3000, 5000, 10000, 30000, 50000]

auc_events = []

for N_tmp in event_sizes:

    energia_tmp = np.random.uniform(0.1, 10, N_tmp)

    logL_tmp = np.random.uniform(
        np.log10(20),
        np.log10(12700),
        N_tmp
    )

    L_tmp = 10**logL_tmp
    delta_tmp = np.random.uniform(0, 2*np.pi, N_tmp)
    phi_tmp = np.random.uniform(0, 2*np.pi, N_tmp)

    P_sm_tmp = probability_sm(energia_tmp, L_tmp,delta_tmp)

    P_nsi_tmp = probability_nsi(energia_tmp, L_tmp, delta_tmp, epsilon=1e-3, phi_nsi=phi_tmp)

    sm_tmp = pd.DataFrame({
        'energia': energia_tmp,
        'baseline': L_tmp,
        'L_over_E': L_tmp/energia_tmp,
        'probabilidade': P_sm_tmp,
        'classe': 0
    })

    nsi_tmp = pd.DataFrame({
        'energia': energia_tmp,
        'baseline': L_tmp,
        'L_over_E': L_tmp/energia_tmp,
        'probabilidade': P_nsi_tmp,
        'classe': 1
    })

    df_tmp = pd.concat([sm_tmp, nsi_tmp])
    X = df_tmp[features]
    y = df_tmp['classe']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42
    )

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )

    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_events.append(auc(fpr, tpr))

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(event_sizes, auc_events, 'o-')
plt.xlabel('Number of events')
plt.ylabel('ROC AUC')
plt.title('Dependence of ROC with the dataset size')
plt.grid()
plt.savefig('plot13.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
noise_levels = [0, 0.001, 0.005, 0.01, 0.05, 0.1]

auc_noise = []

for sigma in noise_levels:

    P_sm_noise = P_sm_events + np.random.normal(0, sigma, N)
    P_nsi_noise = P_nsi_events + np.random.normal(0, sigma, N)

    P_sm_noise = np.clip(P_sm_noise, 0, 1)
    P_nsi_noise = np.clip(P_nsi_noise, 0, 1)

    sm_tmp = pd.DataFrame({
        'energia': energia,
        'baseline': L,
        'L_over_E': L/energia,
        'probabilidade': P_sm_noise,
        'classe': 0
    })

    nsi_tmp = pd.DataFrame({
        'energia': energia,
        'baseline': L,
        'L_over_E': L/energia,
        'probabilidade': P_nsi_noise,
        'classe': 1
    })

    df_tmp = pd.concat([sm_tmp, nsi_tmp])
    X = df_tmp[features]
    y = df_tmp['classe']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )

    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_noise.append(auc(fpr, tpr))

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(noise_levels, auc_noise, 'o-')
plt.xlabel('Experimental Noise')
plt.ylabel('ROC AUC')
plt.title('Impact of the Experimental Noise')
plt.grid()
plt.savefig('plot14.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

In [ ]:
X = df[features]
y = df['classe']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=300, max_depth=12, random_state=42)

rf_model.fit(X_train_scaled, y_train)
rf_probs = rf_model.predict_proba(X_test_scaled)[:,1]
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
rf_auc = auc(rf_fpr, rf_tpr)
print("Random Forest AUC:", rf_auc)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train_scaled, y_train)
xgb_probs = xgb_model.predict_proba(X_test_scaled)[:,1]
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_probs)
xgb_auc = auc(xgb_fpr, xgb_tpr)
print("XGBoost AUC:", xgb_auc)

In [ ]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128,64,32),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    batch_size=256,
    learning_rate='adaptive',
    max_iter=300,
    random_state=42
)

mlp_model.fit(X_train_scaled, y_train)
mlp_probs = mlp_model.predict_proba(X_test_scaled)[:,1]
mlp_fpr, mlp_tpr, _ = roc_curve(y_test, mlp_probs)
mlp_auc = auc(mlp_fpr, mlp_tpr)
print("MLP AUC:", mlp_auc)

In [ ]:
plt.figure(figsize=(9,7))
plt.plot(rf_fpr, rf_tpr, label=f'Random Forest (AUC={rf_auc:.3f})')
plt.plot(xgb_fpr,xgb_tpr, label=f'XGBoost (AUC={xgb_auc:.3f})')
plt.plot(mlp_fpr, mlp_tpr, label=f'MLP Neural Net (AUC={mlp_auc:.3f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Comparasion between Classifiers')
plt.legend()
plt.grid()
plt.savefig('plot15.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
models = ['Random Forest', 'XGBoost', 'MLP']
scores = [rf_auc, xgb_auc, mlp_auc]
plt.figure(figsize=(8,6))
plt.bar(models, scores)
plt.ylabel('ROC AUC')
plt.title('Model Performance')
plt.ylim(0.4,1.0)
plt.grid(axis='y')
plt.savefig('plot16.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
rf_pred = rf_model.predict(X_test_scaled)
cm_rf = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(5,5))
plt.imshow(cm_rf)
plt.title('Random Forest')
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm_rf[i,j],
                 ha='center', va='center')
plt.savefig('plot17.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
xgb_pred = xgb_model.predict(X_test_scaled)
cm_xgb = confusion_matrix(y_test, xgb_pred)
plt.figure(figsize=(5,5))
plt.imshow(cm_xgb)
plt.title('XGBoost')
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm_xgb[i,j],
                 ha='center', va='center')
plt.savefig('plot18.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
mlp_pred = mlp_model.predict(X_test_scaled)
cm_mlp = confusion_matrix(y_test, mlp_pred)
plt.figure(figsize=(5,5))
plt.imshow(cm_mlp)
plt.title('MLP Neural Network')
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm_mlp[i,j],
                 ha='center', va='center')
plt.savefig('plot19.png', dpi = 300, bbox_inches = 'tight')
plt.show()